In [7]:
import torch
import cupy as cp
import open3d as o3d
import numpy as np
import os

def generate_cgh_no_chunking():
    # 1. Konfiguracja sprzętowa (Nvidia RTX 3060Ti)
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    
    # 2. Wczytanie chmury punktów
    filename = r'C:\Users\PC\Downloads\dragon_vrip.ply'
    absolute_path = os.path.abspath(filename)
    pcd = o3d.io.read_point_cloud(absolute_path)
    xyz = np.asarray(pcd.points, dtype=np.float32)
    num_points = xyz.shape[0]
    
    # Przenosimy wszystkie punkty na GPU
    d_pointCloud = torch.tensor(xyz, dtype=torch.float32, device=device)
    print(f'Wczytano {num_points} punktów. Pełna wektoryzacja w toku...')

    # 3. Parametry modulatora SLM Holoeye GAEA 2.0
    px, py = 3840, 2160
    pitch = 3.74e-6
    lambda_val = 632.8e-9
    k = 2 * torch.pi / lambda_val
    N_holograms = 10

    # 4. Generowanie wektorów współrzędnych przestrzennych (PyTorch)
    x_vec = torch.linspace(-px/2, px/2 - 1, px, device=device, dtype=torch.float32) * pitch
    y_vec = torch.linspace(-py/2, py/2 - 1, py, device=device, dtype=torch.float32) * pitch
    Y, X = torch.meshgrid(y_vec, x_vec, indexing='ij')

    # Pre-alokacja macierzy losowych faz dla multipleksowania
    random_phases = torch.rand((py, px, N_holograms), dtype=torch.float32, device=device) * (2 * torch.pi)

    # 5. PEŁNA WEKTORYZACJA 3D (Brak dzielenia na paczki)
    # Zmieniamy kształt chmury punktów, aby wstrzyknąć ją w trzeci wymiar
    px_coord = d_pointCloud[:, 0].view(1, 1, -1)
    py_coord = d_pointCloud[:, 1].view(1, 1, -1)
    pz_coord = d_pointCloud[:, 2].view(1, 1, -1)
    
    # UWAGA: W tym miejscu powstaje gigantyczna macierz o wymiarach [2160, 3840, num_points]
    r_ij = torch.sqrt((X.unsqueeze(-1) - px_coord)**2 + 
                      (Y.unsqueeze(-1) - py_coord)**2 + 
                      pz_coord**2)
    
    # 6. Wyliczenie pola i natychmiastowa redukcja do płaskiego hologramu
    base_hologram = torch.sum(torch.exp(1j * k * r_ij), dim=2)

    # 7. Aplikacja szumów fazowych w ramach multipleksowania czasowego
    holograms_torch = base_hologram.unsqueeze(-1) * torch.exp(1j * random_phases)
    
    # 8. Bezstratny transfer zero-copy do CuPy (DLPack)
    holograms_cupy = cp.from_dlpack(torch.utils.dlpack.to_dlpack(holograms_torch))
    
    print('Generacja zakończona bez pętli!')
    return holograms_cupy

if __name__ == "__main__":
    final_holograms = generate_cgh_no_chunking()

c:\Users\PC\miniconda3\envs\CGH_NED\lib\site-packages\cupy\_environment.py:215: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Wczytano 437645 punktów. Pełna wektoryzacja w toku...


OutOfMemoryError: CUDA out of memory. Tried to allocate 13522.81 GiB. GPU 0 has a total capacity of 8.00 GiB of which 6.61 GiB is free. Of the allocated memory 321.44 MiB is allocated by PyTorch, and 18.56 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)